In [1]:
from pathlib import Path
import numpy as np
import sys

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from spike_classifier import prepare_data, annotate_spikes, train_classifier
from utils.label_utils import get_label_value
from classifier_pipeline.io_utils import load_roi_data

# Use the ROI data that already has labeled ROIs
ROI_DATA_DIR = Path(r"C:\Users\mzinn1\Desktop\gcamp_data")
ROI_DATA_PATH = ROI_DATA_DIR / "all_roi_features.npy"



ROI_DATA_PATH

WindowsPath('C:/Users/mzinn1/Desktop/gcamp_data/all_roi_features.npy')

In [2]:
def summarize_spike_data(npy_path: Path):
    """Summarize spike data from ROI .npy file."""
    if not npy_path.exists():
        print(f"Missing: {npy_path}")
        return
    
    d = np.load(npy_path, allow_pickle=True).item()
    
    rois_with_spikes = sum(1 for roi in d.values() if roi.get('spikes'))
    total_spikes = sum(len(roi.get('spikes', {})) for roi in d.values())
    
    print(f"ROI Data: {npy_path}")
    print(f"Total ROIs: {len(d)}")
    print(f"ROIs with spikes: {rois_with_spikes}")
    print(f"Total spikes: {total_spikes}")
    
    # Count labels from .npy spike dicts
    n_good, n_bad, n_unlabeled = 0, 0, 0
    for roi_data in d.values():
        for spike_data in roi_data.get('spikes', {}).values():
            val = get_label_value(spike_data.get('label', {'value': -1, 'source': 'unlabeled'}))
            if val == 1:
                n_good += 1
            elif val == 0:
                n_bad += 1
            else:
                n_unlabeled += 1
    
    print(f"\nSpike Labels:")
    print(f"  Good: {n_good} | Bad: {n_bad} | Unlabeled: {n_unlabeled}")

summarize_spike_data(ROI_DATA_PATH)

ROI Data: C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features.npy
Total ROIs: 18652
ROIs with spikes: 566
Total spikes: 10203

Spike Labels:
  Good: 356 | Bad: 1005 | Unlabeled: 8842


In [ ]:
from utils.label_utils import create_label_dict

def reset_spike_labels(npy_path: Path):
    """Reset all spike labels to unlabeled."""
    if not npy_path.exists():
        print(f"❌ NPY not found: {npy_path}")
        return
    
    npy_dict = np.load(npy_path, allow_pickle=True).item()
    n_labeled = 0
    for roi_key, roi_data in npy_dict.items():
        if 'spikes' in roi_data:
            for spike_idx in roi_data['spikes']:
                val = get_label_value(roi_data['spikes'][spike_idx].get('label', {}))
                if val != -1:
                    n_labeled += 1
                roi_data['spikes'][spike_idx]['label'] = create_label_dict(-1, 'unlabeled')
    np.save(npy_path, npy_dict, allow_pickle=True)
    
    print(f"✅ Reset {n_labeled} labels to unlabeled")
    print(f"   NPY: {npy_path}")
confirm = input("Are you sure you want to reset all spike labels to unlabeled? (y/n): ")
if confirm.lower() == 'y':
    reset_spike_labels(ROI_DATA_PATH)
    summarize_spike_data(ROI_DATA_PATH)

✅ Reset 1311 labels to unlabeled
   NPY: C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features.npy


KeyboardInterrupt: 

In [3]:
roi_dict = prepare_data.main(
    input_path=str(ROI_DATA_PATH),
    output_path=None,  
    max_rois=None      
)

summarize_spike_data(ROI_DATA_PATH)

Loaded 18652 ROIs from C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features.npy
Processed 569 good ROIs
Skipped 18083 bad ROIs
Preserved 1361 existing labels
Saved to C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features.npy
ROI Data: C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features.npy
Total ROIs: 18652
ROIs with spikes: 569
Total spikes: 10259

Spike Labels:
  Good: 356 | Bad: 1005 | Unlabeled: 8898


In [4]:
from spike_classifier.annotate_spikes import annotate_spikes_by_roi as annotate_spikes

N_ANNOTATIONS = 5000

annotate_spikes(
    data_path=ROI_DATA_PATH,
    unlabeled_only=True
)

Loaded 18652 ROIs from C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features.npy
Saved to C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features.npy
Saved to C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features.npy
Done. ROIs: 553. Labeled spikes: 7 (updated=7, confirmed=0, skipped=0).


SessionStats(rois_total=553, rois_done=167, spikes_total_in_session=0, spikes_labeled=7, spikes_updated=7, spikes_confirmed=0, spikes_skipped=0)

In [5]:
# Cell 5: Check Label Distribution
summarize_spike_data(ROI_DATA_PATH)

# Check class balance from .npy
d = np.load(ROI_DATA_PATH, allow_pickle=True).item()
good = sum(1 for roi in d.values() for s in roi.get('spikes', {}).values() if get_label_value(s.get('label', {})) == 1)
bad = sum(1 for roi in d.values() for s in roi.get('spikes', {}).values() if get_label_value(s.get('label', {})) == 0)
print(f"\nGood/Bad ratio: {good} / {bad}")

if good < 10 or bad < 10:
    print("⚠️ Warning: Need more labeled samples for training!")

ROI Data: C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features.npy
Total ROIs: 18652
ROIs with spikes: 569
Total spikes: 10259

Spike Labels:
  Good: 358 | Bad: 1010 | Unlabeled: 8891

Good/Bad ratio: 358 / 1010


In [6]:

from spike_classifier.train_classifier import train_spike_classifier

MODEL_OUT_DIR = Path(r"C:\Users\mzinn1\Desktop\gcamp_model")
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

config_path = PROJECT_ROOT / "config" / "classifier_config.yaml"

results = train_spike_classifier(
    config_path=config_path,
    data_path=ROI_DATA_PATH,
    name="spike_classifier",
    output_dir=MODEL_OUT_DIR,
    verbose=True,
    manual_only=True
)

print(f"\n🏆 Best model: {type(results.model).__name__}")
print(f"Transform: {results.transform}")
print(f"Test accuracy: {results.test_acc:.4f}")
print(f"ROC AUC: {results.roc_auc:.4f}")
print(f"Selected features: {results.features}")

Loaded 18652 ROIs from C:\Users\mzinn1\Desktop\gcamp_data\all_roi_features.npy
Dataset Summary
Total labeled ROIs: 1368
  Train: 1094 | Test: 274

Feature names:
	  1. spike_prom
	  2. dominance_score
	  3. mini_prom
	  4. distance

Label distribution:
              Bad (0)  Good (1)
  Train           813       281
  Test            197        77
  Total          1010       358

Training on: Manual labels only
RF
	 Best transform: sqrt with roc_auc: 0.9728
RF with transform: sqrt
	 Best features: ['spike_prom', 'mini_prom', 'dominance_score', 'distance'] with roc_auc: 0.9716
LR
	 Best transform: sqrt with roc_auc: 0.9794
LR with transform: sqrt
	 Best features: ['spike_prom', 'mini_prom', 'distance'] with roc_auc: 0.9794


c:\Users\mzinn1\AppData\Local\anaconda3\envs\gcamp\lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



--------------------------------------------------
TUNED MODEL SUMMARY
--------------------------------------------------
Model:     LogisticRegression
Transform: sqrt
Features:  ['spike_prom', 'mini_prom', 'distance']

Hyperparameters:
  C: 10
  class_weight: None
  max_iter: 200
  penalty: l1
  solver: saga

Metrics:
  CV Accuracy:   0.9634
  Test Accuracy: 0.9635
  ROC AUC:       0.9790
  F1:            0.9632
  Precision:     0.9634
  Recall:        0.9635

Confusion Matrix:
              Pred 0  Pred 1
  Actual 0    194     3      
  Actual 1    7       70     
--------------------------------------------------
Saved model to C:\Users\mzinn1\Desktop\gcamp_model\spike_classifier.joblib
Saved results to C:\Users\mzinn1\Desktop\gcamp_model\spike_classifier_results.json
Saved results to C:\Users\mzinn1\Desktop\gcamp_model

🏆 Best model: LogisticRegression
Transform: sqrt
Test accuracy: 0.9635
ROC AUC: 0.9790
Selected features: ['spike_prom', 'mini_prom', 'distance']


c:\Users\mzinn1\AppData\Local\anaconda3\envs\gcamp\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
